# Top-NMF: Example 2
Graph topological NMF with built-in utilities.

In [ ]:
import numpy as np
import torch
from itertools import combinations
import matplotlib.pyplot as plt

from TopNMF import TopologicalNMF, GraphFiltrationPH, plot_gallery_graph, plot_loss, plot_PD_graph


In [ ]:
def generate_clique_dataset(
    cliques=((0, 1, 2, 3, 4), (4, 5, 6), (5, 6, 7, 8)),
    weights=((2, 1, 2), (1, 2, 1), (1, 1, 2)),
):
    all_nodes = sorted({i for clique in cliques for i in clique})
    edge_list = [tuple(sorted(e)) for e in combinations(all_nodes, 2)]
    edge_index = {e: i for i, e in enumerate(edge_list)}
    X = []

    for alpha in weights:
        row = np.zeros(len(edge_list))
        for coeff, clique in zip(alpha, cliques):
            for u, v in combinations(clique, 2):
                row[edge_index[tuple(sorted((u, v)))]] += coeff
        X.append(row)

    return np.stack(X), edge_list

X, all_edges = generate_clique_dataset()
print(f"X shape: {X.shape}, edges: {len(all_edges)}")


In [ ]:
device = 'cpu'
n_components = 3

ph_complex = GraphFiltrationPH(max_dim=1, superlevel=False)
model = TopologicalNMF(
    n_components=n_components,
    device=device,
    random_state=43,
    complex=ph_complex,
    use_embedding=False,
)

model.fit(
    X,
    n_iterations=300,
    lr=0.005,
    lambda_top=0.05,
    PH_dims=[0, 1],
    complex_inputs={'all_edges': all_edges},
    verbose=True,
)


In [ ]:
edge_index = torch.tensor(all_edges, dtype=torch.long).T
fig, ax = plt.subplots(1, 1, figsize=(4, 3))
plot_loss(model.losses, ax=ax)
ax.set_title('Loss')
plt.tight_layout()
plt.show()

basis = model.V.detach().cpu()
fig, axs = plt.subplots(2, 2, figsize=(6, 6))
plot_gallery_graph(basis, edge_index, title='Basis', n_col=2, n_row=2, axs=axs)
plt.tight_layout()
plt.show()

plot_PD_graph(basis, all_edges, n_col=2, n_row=2, superlevel=False)
plt.show()
